# Financial Fraud Detection: Guided Dataset Review

This notebook uses the supplied synthetic transaction file. `Fraud_Label` is the provided label, not independently verified real-world fraud. Run cells from top to bottom. The raw CSV is never overwritten.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import load_dataset, clean_transactions, profile_dataset
from src.eda import run_eda

raw = load_dataset()
cleaned, audit = clean_transactions(raw)
print(f"Loaded {len(raw):,} rows and {len(raw.columns)} columns.")
print(f"Target column: Fraud_Label; fraud-label rate: {cleaned['Fraud_Label'].mean():.1%}")

Loaded 50,000 rows and 14 columns.
Target column: Fraud_Label; fraud-label rate: 32.1%


## 1. Check columns and data quality

The table below gives the data type, missing-cell count, and number of unique values for each column. The audit also records duplicate removal and target/date checks.

In [2]:
display(pd.DataFrame(profile_dataset(cleaned)["column_details"]))
print(audit)
display(cleaned["Fraud_Label"].value_counts().rename(index={0: "Legitimate label (0)", 1: "Fraud label (1)"}))

,name,dtype,missing,unique
0,Transaction_ID,string,0,50000
1,User_ID,string,0,8963
2,Transaction_Amount,float64,0,21763
3,Transaction_Type,string,0,4
4,Date,datetime64[us],0,365
5,Account_Balance,float64,0,49867
6,Device_Type,string,0,3
7,Location,string,0,5
8,Merchant_Category,string,0,5
9,Previous_Fraudulent_Activity,int64,0,2


{'rows_before_cleaning': 50000, 'rows_after_cleaning': 50000, 'duplicate_rows_removed': 0, 'rows_without_valid_target_removed': 0, 'missing_values_by_column': {}, 'unparseable_numeric_values_by_column': {}, 'negative_transaction_amounts': 0, 'negative_card_ages': 0, 'date_min': '2023-01-01', 'date_max': '2023-12-31', 'target_distribution': {'0': 33933, '1': 16067}}


Fraud_Label
Legitimate label (0)    33933
Fraud label (1)         16067
Name: count, dtype: int64

## 2. Generate exploratory charts

Charts are saved under `reports/figures/`. They describe this sample only and do not prove that a transaction pattern causes fraud.

In [3]:
insights = run_eda(cleaned)
print(insights)
print("Open reports/figures/ to see the saved charts.")

{'rows': 50000, 'fraud_labeled_transactions': 16067, 'legitimate_labeled_transactions': 33933, 'fraud_rate': 0.32134, 'total_transaction_value': 4970550.609999999, 'mean_transaction_amount': 99.41101219999999, 'daily_transaction_volume_peak': 170, 'fraud_rate_by_transaction_type': {'ATM Withdrawal': 0.32490163012928613, 'Bank Transfer': 0.3208319948602634, 'Online': 0.3212976247409533, 'POS': 0.31835205992509363}, 'missing_values_by_column': {}, 'interpretation_note': 'These are associations in the supplied synthetic dataset; they do not establish causation or real-world fraud performance.'}
Open reports/figures/ to see the saved charts.


## 3. Read the model report

Run `python -m src.train_model` from the project root before this cell. The project uses a time-based holdout so later transactions are not mixed into model training. The supplied synthetic labels currently show near-chance performance; the candidate model is for learning and review only.

In [4]:
metrics_path = PROJECT_ROOT / "reports" / "model_metrics.json"
if metrics_path.exists():
    import json
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(metrics["model_comparison"]).T)
    print("Qualified for real-world use:", metrics["model_qualified_for_real_world_use"])
else:
    print("Model metrics are not available yet. Run: python -m src.train_model")

,accuracy,precision,recall,f1,roc_auc,pr_auc,confusion_matrix
Logistic Regression,0.557904,0.322046,0.336634,0.329178,0.496549,0.31888,"[[5635, 2863], [2680, 1360]]"
Decision Tree,0.520019,0.319196,0.432178,0.367192,0.497388,0.318056,"[[4774, 3724], [2294, 1746]]"
Random Forest,0.638459,0.313681,0.102723,0.154764,0.494853,0.317399,"[[7590, 908], [3625, 415]]"


Qualified for real-world use: False
